# Data cleaning

In [17]:
import ast

import pandas as pd

DATA = 'data'

movies   = pd.read_csv(f'{DATA}/movies_metadata.csv', low_memory=False)
credits  = pd.read_csv(f'{DATA}/credits.csv')
keywords = pd.read_csv(f'{DATA}/keywords.csv')
links    = pd.read_csv(f'{DATA}/links.csv')
ratings  = pd.read_csv(f'{DATA}/ratings.csv',
                       dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'})

rows_before = {'movies': len(movies), 'credits': len(credits), 'keywords': len(keywords),
               'links': len(links), 'ratings': len(ratings)}
deleted = {}
edited = {}

for name, n in rows_before.items():
    print(f'Loaded {n:,} rows from {name}')

Loaded 45,466 rows from movies
Loaded 45,476 rows from credits
Loaded 46,419 rows from keywords
Loaded 45,843 rows from links
Loaded 26,024,289 rows from ratings


## Movies: broken rows
3 movies are split over two rows. Both halves are removed.

In [18]:
before = len(movies)
bad_idx = movies.index[~movies['id'].astype(str).str.fullmatch(r'\d+')]
movies = movies.drop(index=list(bad_idx) + list(bad_idx - 1))

deleted['Movies: broken rows'] = before - len(movies)
print(f"Deleted {deleted['Movies: broken rows']:,} rows")
print(f'Rows in movies: {len(movies):,}')

Deleted 6 rows
Rows in movies: 45,460


## Movies: data types
Convert text columns to numbers, dates and booleans.

In [19]:
movies['id'] = movies['id'].astype('int64')
movies['budget'] = pd.to_numeric(movies['budget'])
movies['popularity'] = pd.to_numeric(movies['popularity'])
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['vote_count'] = movies['vote_count'].astype('Int64')
to_bool = {'True': True, 'False': False, True: True, False: False}
movies['adult'] = movies['adult'].map(to_bool).astype('boolean')
movies['video'] = movies['video'].map(to_bool).astype('boolean')

edited['Movies: data types'] = len(movies)
print(f"Edited {len(movies):,} rows (id, budget, popularity, vote_count -> numbers, release_date -> datetime, adult, video -> bool)")
print(movies[['id', 'budget', 'popularity', 'vote_count', 'release_date', 'adult', 'video']].dtypes.to_string())

Edited 45,460 rows (id, budget, popularity, vote_count -> numbers, release_date -> datetime, adult, video -> bool)
id                       int64
budget                   int64
popularity             float64
vote_count               Int64
release_date    datetime64[us]
adult                  boolean
video                  boolean


## Movies: duplicates
Remove exact copies. For other duplicated ids, keep the row with the most votes. Must run before parsing nested columns.

In [20]:
before = len(movies)
movies = movies.drop_duplicates()
deleted['Movies: exact duplicates'] = before - len(movies)

before = len(movies)
movies = (movies.sort_values('vote_count', ascending=False)
                .drop_duplicates('id')
                .sort_index())
deleted['Movies: duplicate ids'] = before - len(movies)

print(f"Deleted {deleted['Movies: exact duplicates']:,} exact duplicates and {deleted['Movies: duplicate ids']:,} duplicate ids")
print(f'Rows in movies: {len(movies):,} | ids still duplicated: {movies["id"].duplicated().sum()}')

Deleted 17 exact duplicates and 13 duplicate ids
Rows in movies: 45,430 | ids still duplicated: 0


## Movies: unknown values
0 in budget, revenue and runtime means unknown, so set to missing.

In [21]:
UNKNOWN_IF_ZERO = ['budget', 'revenue', 'runtime']

changed = 0
for col in UNKNOWN_IF_ZERO:
    zeros = movies[col] == 0
    print(f'  {col:8s} {zeros.sum():>7,} zeros -> missing')
    changed |= zeros
    movies[col] = movies[col].mask(zeros).astype('float64')

edited['Movies: unknown values'] = int(changed.sum())
print(f"Edited {edited['Movies: unknown values']:,} rows")

  budget    36,550 zeros -> missing
  revenue   38,032 zeros -> missing
  runtime    1,558 zeros -> missing
Edited 40,066 rows


## Movies: unrealistic budgets and revenues
Budgets and revenues under $1,000 are set to missing.

In [22]:
MIN_AMOUNT = 1000

for col in ['budget', 'revenue']:
    too_low = movies[col] < MIN_AMOUNT   # missing values are not counted
    print(f'{col}: {too_low.sum():,} below ${MIN_AMOUNT:,} | examples: {movies.loc[too_low, col].value_counts().head(6).to_dict()}')
    movies.loc[too_low, col] = None
    edited[f'Movies: {col} < $1,000'] = int(too_low.sum())

print(f"Edited {edited['Movies: budget < $1,000']:,} budgets and {edited['Movies: revenue < $1,000']:,} revenues")

budget: 244 below $1,000 | examples: {1.0: 25, 2.0: 14, 10.0: 10, 3.0: 9, 5.0: 8, 500.0: 8}
revenue: 154 below $1,000 | examples: {1.0: 12, 3.0: 9, 5.0: 5, 15.0: 5, 8.0: 5, 7.0: 4}
Edited 244 budgets and 154 revenues


## Movies: nested columns
Parse text into lists/dicts.

In [23]:
NESTED_COLS = ['genres', 'production_companies', 'production_countries', 'spoken_languages', 'belongs_to_collection']

def parse(x):
    return ast.literal_eval(x) if isinstance(x, str) else None

for col in NESTED_COLS:
    movies[col] = movies[col].apply(parse)

edited['Movies: nested columns'] = len(movies)
print(f'Parsed {len(NESTED_COLS)} columns in {len(movies):,} rows')

Parsed 5 columns in 45,430 rows


## Credits: duplicates
Remove exact copies, then keep one row per id.

In [24]:
before = len(credits)
credits = credits.drop_duplicates()
deleted['Credits: exact duplicates'] = before - len(credits)

before = len(credits)
credits = credits.drop_duplicates('id')
deleted['Credits: duplicate ids'] = before - len(credits)

print(f"Deleted {deleted['Credits: exact duplicates']:,} exact duplicates and {deleted['Credits: duplicate ids']:,} duplicate ids")
print(f'Rows in credits: {len(credits):,}')

Deleted 37 exact duplicates and 7 duplicate ids
Rows in credits: 45,432


## Credits: nested columns

In [25]:
credits['cast'] = credits['cast'].apply(ast.literal_eval)
credits['crew'] = credits['crew'].apply(ast.literal_eval)

edited['Credits: nested columns'] = len(credits)
print(f'Parsed cast and crew in {len(credits):,} rows')

Parsed cast and crew in 45,432 rows


## Keywords: duplicates
All duplicates are exact copies. Keep one row per id.

In [26]:
before = len(keywords)
keywords = keywords.drop_duplicates('id')

deleted['Keywords: duplicate ids'] = before - len(keywords)
print(f"Deleted {deleted['Keywords: duplicate ids']:,} rows")
print(f'Rows in keywords: {len(keywords):,}')

Deleted 987 rows
Rows in keywords: 45,432


## Keywords: nested column

In [27]:
keywords['keywords'] = keywords['keywords'].apply(ast.literal_eval)

edited['Keywords: nested column'] = len(keywords)
print(f'Parsed keywords in {len(keywords):,} rows')

Parsed keywords in 45,432 rows


## Links
Remove links without a `tmdbId` or whose `tmdbId` is not in movies.

In [28]:
movie_ids = set(movies['id'])

before = len(links)
links = links.dropna(subset=['tmdbId'])
deleted['Links: missing tmdbId'] = before - len(links)
links['tmdbId'] = links['tmdbId'].astype('int64')

before = len(links)
links = links[links['tmdbId'].isin(movie_ids)]
deleted['Links: tmdbId not in movies'] = before - len(links)

print(f"Deleted {deleted['Links: missing tmdbId']:,} rows without tmdbId and {deleted['Links: tmdbId not in movies']:,} not found in movies")
print(f"Duplicated movieId: {links['movieId'].duplicated().sum()} | tmdbId shared by several movieIds: {links['tmdbId'].duplicated().sum()}")
print(f'Rows in links: {len(links):,}')

Deleted 219 rows without tmdbId and 164 not found in movies
Duplicated movieId: 0 | tmdbId shared by several movieIds: 30
Rows in links: 45,460


## Ratings
Add `movie_id` (the movie's `id`) through links, remove ratings without a movie, and convert `timestamp` to datetime.

In [29]:
print('Missing values:', int(ratings.isna().sum().sum()),
      '| same user rating same movie twice:', ratings.duplicated(['userId', 'movieId']).sum())

ratings['movie_id'] = ratings['movieId'].map(links.set_index('movieId')['tmdbId'])

before = len(ratings)
ratings = ratings.dropna(subset=['movie_id'])
deleted['Ratings: no matching movie'] = before - len(ratings)
ratings['movie_id'] = ratings['movie_id'].astype('int64')

ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
edited['Ratings: movie_id + datetime'] = len(ratings)

print(f"Deleted {deleted['Ratings: no matching movie']:,} ratings without a matching movie")
print(f'Rows in ratings: {len(ratings):,} | users: {ratings["userId"].nunique():,} | movies: {ratings["movie_id"].nunique():,}')
ratings.head(3)

Missing values: 0 | same user rating same movie twice: 0
Deleted 42,722 ratings without a matching movie
Rows in ratings: 25,981,567 | users: 270,883 | movies: 44,708


,userId,movieId,rating,timestamp,movie_id
0,1,110,1.0,2015-03-09 22:52:09,197
1,1,147,4.5,2015-03-09 23:07:15,10474
2,1,858,5.0,2015-03-09 22:52:03,238


## Prepare for MongoDB
Turn missing values into `None` so they are stored as null. Ratings are converted in batches.

In [30]:
def to_documents(df):
    # Plain Python objects, with every missing value as None
    return df.astype(object).where(df.notna(), None).to_dict('records')

def batches(df, size=100_000):
    for start in range(0, len(df), size):
        yield to_documents(df.iloc[start:start + size])

movie_docs   = to_documents(movies)
credit_docs  = to_documents(credits)
keyword_docs = to_documents(keywords)

def has_nan(docs):
    return sum(isinstance(v, float) and v != v for d in docs for v in d.values())

for name, docs in [('movies', movie_docs), ('credits', credit_docs), ('keywords', keyword_docs)]:
    print(f'{name:9s} {len(docs):>7,} documents | NaN left: {has_nan(docs)}')
print('ratings  ', f'{len(ratings):,} rows, missing values: {int(ratings.isna().sum().sum())} (converted in batches of 100,000)')

print('\nExample movie with unknown values:')
example = next(d for d in movie_docs if d['budget'] is None and d['release_date'] is None)
print({k: example[k] for k in ['id', 'title', 'budget', 'revenue', 'runtime', 'release_date', 'vote_count', 'adult']})

movies     45,430 documents | NaN left: 0
credits    45,432 documents | NaN left: 0
keywords   45,432 documents | NaN left: 0
ratings   25,981,567 rows, missing values: 0 (converted in batches of 100,000)

Example movie with unknown values:
{'id': 365371, 'title': 'War Stories Our Mother Never Told Us', 'budget': None, 'revenue': None, 'runtime': 95.0, 'release_date': None, 'vote_count': 0, 'adult': False}


## Summary

In [31]:
rows_after = {'movies': len(movies), 'credits': len(credits), 'keywords': len(keywords),
              'links': len(links), 'ratings': len(ratings)}
print(f"{'':10} {'before':>11} {'after':>11}")
for name in rows_before:
    print(f'{name:10} {rows_before[name]:>11,} {rows_after[name]:>11,}')

print('\nDeleted')
for step, n in deleted.items():
    print(f'  {step:<30} {n:>11,}')
print('\nEdited')
for step, n in edited.items():
    print(f'  {step:<30} {n:>11,}')

                before       after
movies          45,466      45,430
credits         45,476      45,432
keywords        46,419      45,432
links           45,843      45,460
ratings     26,024,289  25,981,567

Deleted
  Movies: broken rows                      6
  Movies: exact duplicates                17
  Movies: duplicate ids                   13
  Credits: exact duplicates               37
  Credits: duplicate ids                   7
  Keywords: duplicate ids                987
  Links: missing tmdbId                  219
  Links: tmdbId not in movies            164
  Ratings: no matching movie          42,722

Edited
  Movies: data types                  45,460
  Movies: unknown values              40,066
  Movies: budget < $1,000                244
  Movies: revenue < $1,000               154
  Movies: nested columns              45,430
  Credits: nested columns             45,432
  Keywords: nested column             45,432
  Ratings: movie_id + datetime    25,981,567


## Save cleaned data
Save to `data/clean/` as pickle. Use `to_documents`/`batches` before inserting.

In [32]:
import os

OUT = f'{DATA}/clean'
os.makedirs(OUT, exist_ok=True)

for name, df in [('movies', movies), ('credits', credits), ('keywords', keywords), ('links', links), ('ratings', ratings)]:
    path = f'{OUT}/{name}.pkl'
    df.to_pickle(path)
    print(f'Saved {name:9s} {len(df):>11,} rows -> {path} ({os.path.getsize(path) / 1e6:,.0f} MB)')

Saved movies         45,430 rows -> data/clean/movies.pkl (37 MB)
Saved credits        45,432 rows -> data/clean/credits.pkl (172 MB)
Saved keywords       45,432 rows -> data/clean/keywords.pkl (6 MB)
Saved links          45,460 rows -> data/clean/links.pkl (1 MB)
Saved ratings    25,981,567 rows -> data/clean/ratings.pkl (935 MB)
